In [ ]:
from brian2 import *
from brian2tools import *
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import imageio
import os
from scipy.signal import convolve2d
from pathlib import Path
from progressbar import progressbar
from scipy.signal import butter, filtfilt, firwin
set_device('cpp_standalone', build_on_run=False)
results_PATH = Path('results_standalone') 

start_scope()

def build_network(tau_ins_AMPA_E=5000*second,
                  tau_int_AMPA_E=500*second,
                  tau_ins_GABA_E=5000*second,
                  tau_int_GABA_E=500*second,
                  tau_HSP=500*second,
                  target_rate=2.0*Hz):
    alcohol=0
    n_rc=10
    n_input_rows = n_rc  
    n_input_columns = n_rc 
    n_output_rows = n_rc  
    n_output_columns = n_rc 
    N_inp = n_input_rows*n_input_columns
    N_E = n_output_rows*n_output_columns
    n_inh_rows, n_inh_cols = 5, 5
    N_I = n_inh_rows * n_inh_cols

    # Inhibitory neuron positions (scaled to excitatory grid)
    inh_positions = []
    for iy in range(n_inh_rows):
        for ix in range(n_inh_cols):
            # scale inhibitory coordinates to excitatory grid
            x = int((ix + 0.5) * n_output_columns / n_inh_cols)
            y = int((iy + 0.5) * n_output_rows / n_inh_rows)
            inh_positions.append((x, y))


    # voltage
    V_L = -70. * mV
    V_thr = -50. * mV
    V_reset = -60. * mV
    V_E = 0.0 * mV
    V_I = -70. * mV

    # membrane capacitance
    C_m_E = 0.5 * nF
    C_m_I = 0.2 * nF

    # membrane leak
    g_m_E = 10. * nS
    g_m_I = 15. * nS

    # refractory period
    tau_rp_E = 2. * ms
    tau_rp_I = 1. * ms

    # AMPA
    g_AMPA_ext_E = 5.7 * nS 
    g_AMPA_rec_E = 5.7 * nS 
    g_AMPA_ext_I = 5.7 * nS
    g_AMPA_rec_I = 5.7 * nS 
    tau_AMPA = 8. * ms


    # GABAa
    g_GABA_E = 1.0*nS 
    g_GABA_I = 1.0*nS 
    tau_GABA = 10. * ms



    #STDP/iSTDP
    Apre   =  0.0025
    Apost  = -0.0030
    w_max = 1.0 #1.5

    Apre_i  = -0.0010
    Apost_i =  0.0012
    w_max_i = 1.0


    tau_HSP = 500*second           
    target_rate = 2.0*Hz            
    record_dt = 200*ms

    tau_vm = 10*second
    tau_active = 10*second
    

    center_neurons = [14, 24,34, 44, 54, 64,89]
    recurrent_radius = 3

    # Alcohol effect strengths (fractions)
    ALC_RED_AMPA_E = 0.1   # X% reduction of excitatory drive onto E
    ALC_RED_AMPA_I = 0.5   # X% increase of excitatory drive onto I
    ALC_INC_GABA_E = 0.2   # X% increase of inhibitory drive onto E
    ALC_INC_GABA_I = 0.2   # X% decrease of inhibitory drive onto I



    eqs_E = '''
    dv / dt = (- g_m_E * (v - V_L) - I_syn ) / C_m_E : volt (unless refractory)
    I_syn = I_AMPA_ext + I_AMPA_rec + I_GABA_rec : amp

    # Homeostatically scaled synaptic currents
    I_AMPA_ext = g_AMPA_ext_E * (v - V_E) * s_AMPA_ext : amp
    I_AMPA_rec = g_AMPA_rec_E * g_AMPA_E_hsp * (v - V_E) * s_AMPA : amp
    I_GABA_rec = g_GABA_E * g_GABA_E_hsp * (v - V_I) * s_GABA     : amp

    ds_AMPA_ext / dt = - s_AMPA_ext / tau_AMPA : 1
    ds_AMPA / dt      = - s_AMPA / tau_AMPA     : 1
    ds_GABA / dt      = - s_GABA / tau_GABA     : 1

    dx_spike / dt = -x_spike/tau_HSP : 1
    r_est = x_spike / tau_HSP : Hz

    # HSP scaling variables (dimensionless)
    dg_AMPA_E_hsp/dt = ((1 - g_AMPA_E_hsp) * clip((target_rate - r_est)/target_rate, 0, 1) / tau_ins_AMPA_E
                - (g_AMPA_E_hsp - 0.1) * clip((r_est - target_rate)/target_rate, 0, 1) / tau_int_AMPA_E) : 1

    dg_GABA_E_hsp/dt = (-(g_GABA_E_hsp - 0.1) * clip((target_rate - r_est)/target_rate, 0, 1) / tau_int_GABA_E
                + (1 - g_GABA_E_hsp) * clip((r_est - target_rate)/target_rate, 0, 1) / tau_ins_GABA_E) : 1

    dvm_lp / dt = (v - vm_lp)/tau_vm : volt
    dy_act / dt = -y_act / tau_active : 1

    alc = int(t >= t_on) * int(t < t_off) : 1
    t_on  : second (shared)
    t_off : second (shared)

    tau_ins_AMPA_E : second (shared)
    tau_int_AMPA_E : second (shared)
    tau_int_GABA_E : second (shared)
    tau_ins_GABA_E : second (shared)
    '''


    eqs_I = '''
    dv / dt = (- g_m_I * (v - V_L) - I_syn) / C_m_I : volt (unless refractory)
    I_syn = I_AMPA_ext + I_AMPA_rec + I_GABA_rec : amp
    I_AMPA_ext = g_AMPA_ext_I * (1 - ALC_RED_AMPA_I * alc) * (v - V_E) * s_AMPA_ext : amp
    I_AMPA_rec = g_AMPA_rec_I * g_AMPA_I_hsp * (1 + ALC_RED_AMPA_I * alc) * (v - V_E) * 1 * s_AMPA : amp
    ds_AMPA_ext / dt = - s_AMPA_ext / tau_AMPA : 1
    ds_AMPA / dt = - s_AMPA / tau_AMPA : 1

    I_GABA_rec = g_GABA_I * g_GABA_I_hsp * (1 + ALC_INC_GABA_I * alc) * (v - V_I) * s_GABA : amp
    ds_GABA / dt = - s_GABA / tau_GABA : 1

    dx_spike / dt = -x_spike/tau_HSP : 1
    r_est = x_spike / tau_HSP : Hz

    dg_AMPA_I_hsp/dt = ((1 - g_AMPA_I_hsp) * clip((-r_est +target_rate)/target_rate, 0, 1) / tau_ins_AMPA_I
                - (g_AMPA_I_hsp - 0.1) * clip((-target_rate + r_est)/target_rate, 0, 1) / tau_int_AMPA_I) : 1

    dg_GABA_I_hsp/dt = (-(g_GABA_I_hsp - 0.1) * clip((-r_est + target_rate)/target_rate, 0, 1) / tau_int_GABA_I
                + (1 - g_GABA_I_hsp) * clip((-target_rate + r_est)/target_rate, 0, 1) / tau_ins_GABA_I) : 1


    dy_act / dt = -y_act / tau_active : 1

    alc = int(t >= t_on) * int(t < t_off) : 1
    t_on  : second (shared)
    t_off : second (shared)

    tau_ins_AMPA_I : second (shared)
    tau_int_AMPA_I : second (shared)
    tau_int_GABA_I : second (shared)
    tau_ins_GABA_I : second (shared)
    '''

    P_E = NeuronGroup(N_E, eqs_E, threshold='v > V_thr',
                      reset='v = V_reset; x_spike += 1.; y_act = 1.',
                      refractory=tau_rp_E, method='euler')
    P_E.v = V_L
    P_E.g_AMPA_E_hsp = 0.3  
    P_E.g_GABA_E_hsp = 0.3     
    P_E.tau_ins_AMPA_E = tau_ins_AMPA_E
    P_E.tau_int_AMPA_E = tau_int_AMPA_E
    P_E.tau_ins_GABA_E = tau_ins_GABA_E
    P_E.tau_int_GABA_E = tau_int_GABA_E

    P_I = NeuronGroup(N_I, eqs_I, threshold='v > V_thr',
                      reset='v = V_reset; x_spike += 1.; y_act = 1.',
                      refractory=tau_rp_I, method='euler')
    P_I.v = V_L
    P_I.g_AMPA_I_hsp = 0.3  
    P_I.g_GABA_I_hsp = 0.3    
    P_I.tau_ins_AMPA_I = 500000000000000*second
    P_I.tau_int_AMPA_I = 500000000000000*second
    P_I.tau_ins_GABA_I = 500000000000000*second
    P_I.tau_int_GABA_I = 500000000000000*second
    
    pg_inp = PoissonGroup(N_inp, rates=0 * Hz, name = 'inp')
    C_PG_E = Synapses(pg_inp, P_E, model='w:1', on_pre = '''
            s_AMPA_ext += w  
            ''', method='euler', name='PG_E')
    C_PG_E.connect(j='i')  

    
    C_E_E  = Synapses(P_E, P_E, model='w:1', on_pre = '''
        s_AMPA += w  
        ''', method='euler', name='E_E')
    recurrent_radius = 2
    for y1 in range(n_output_rows):
        for x1 in range(n_output_columns):
            post = y1 * n_output_columns + x1
            for dy in range(-recurrent_radius, recurrent_radius + 1):
                for dx in range(-recurrent_radius, recurrent_radius + 1):
                    if dx == 0 and dy == 0:
                        continue
                    x2 = x1 + dx
                    y2 = y1 + dy
                    if 0 <= x2 < n_output_columns and 0 <= y2 < n_output_rows:
                        pre = y2 * n_output_columns + x2
                        C_E_E.connect(i=pre, j=post)
    C_E_I  = Synapses(P_E, P_I, model='w:1', on_pre='''
    s_AMPA += w
    ''', method='euler', name='E_I') #        x_postE_post += 1.0 
    C_I_E  = Synapses(P_I, P_E, model='w:1', on_pre='s_GABA += w', method='euler', name='I_E')
    for inh_idx, (ix, iy) in enumerate(inh_positions):
        for ex_y in range(n_output_rows):
            for ex_x in range(n_output_columns):
                if abs(ex_x - ix) <= recurrent_radius and abs(ex_y - iy) <= recurrent_radius:
                    exc_idx = ex_y * n_output_columns + ex_x
                    C_I_E.connect(i=inh_idx, j=exc_idx)
                    C_E_I.connect(i=exc_idx, j=inh_idx)
    C_I_I  = Synapses(P_I, P_I, model='w:1', on_pre='s_GABA += w', method='euler', name='I_I')
    for iy in range(n_inh_rows):
        for ix in range(n_inh_cols):
            post = iy * n_inh_cols + ix
            for dy in range(-1, 2):
                for dx in range(-1, 2):
                    if dx == 0 and dy == 0:
                        continue
                    x2, y2 = ix + dx, iy + dy
                    if 0 <= x2 < n_inh_cols and 0 <= y2 < n_inh_rows:
                        pre = y2 * n_inh_cols + x2
                        C_I_I.connect(i=pre, j=post)

    C_PG_E.w[:] = 1.15 
    C_I_E.w[:] = 0.3
    C_I_I.w[:] = 0.3
    C_E_I.w[:] = 0.3
    C_E_E.w[:] = 0.4
    vm_container = NeuronGroup(1, 'vm_pop : volt', method='euler')
    C_vm = Synapses(P_E, vm_container,
                model='vm_pop_post = vm_lp_pre / N_pre : volt (summed)', 
                method='euler')
    C_vm.connect()

    act_container = NeuronGroup(1, 'frac_act : 1', method='euler')
    
    C_act = Synapses(
        P_E, act_container,
        model='''scale:1 (constant)
            frac_act_post = scale * clip(y_act_pre, 0, 1) : 1 (summed)''',
        method='euler'
    )
    C_act.connect()
    C_act.scale = 1.0/float(N_E)

    act_cont_I = NeuronGroup(1, 'frac_act : 1', method='euler')
    
    C_act_I = Synapses(
        P_I, act_cont_I,
        model='''scale:1 (constant)
            frac_act_post = scale * clip(y_act_pre, 0, 1) : 1 (summed)''',
        method='euler'
    )
    C_act_I.connect()
    C_act_I.scale = 1.0/float(N_I)
    
    rate_container_E = NeuronGroup(1, 'r_pop_E : Hz', method='euler')
    C_rE = Synapses(P_E, rate_container_E,
                model='r_pop_E_post = r_est_pre / N_pre : Hz (summed)',
                method='euler')
    C_rE.connect()

    rate_container_I = NeuronGroup(1, 'r_pop_I : Hz', method='euler')
    C_rI = Synapses(P_I, rate_container_I,
                model='r_pop_I_post = r_est_pre / N_pre : Hz (summed)',
                method='euler')
    C_rI.connect()

    # monitors
    V_mon = StateMonitor(vm_container, 'vm_pop', record=True, dt=record_dt,name='V_mon')
    act_mon = StateMonitor(act_container, 'frac_act', record=True, dt=record_dt, name='act_mon')
    act_mon_I = StateMonitor(act_cont_I, 'frac_act', record=True, dt=record_dt, name='act_mon_I')
    
    rate_pop_mon_E = StateMonitor(rate_container_E, 'r_pop_E', record=True, dt=record_dt, name='rate_pop_mon_E')
    rate_pop_mon_I = StateMonitor(rate_container_I, 'r_pop_I', record=True, dt=record_dt, name='rate_pop_mon_I')

    gAMPA_mon_E = StateMonitor(P_E, 'g_AMPA_E_hsp', record=True, dt=record_dt, name='g_AMPA_E_hsp')
    gGABA_mon_E = StateMonitor(P_E, 'g_GABA_E_hsp', record=True, dt=record_dt, name='g_GABA_E_hsp')
    gAMPA_mon_I = StateMonitor(P_I, 'g_AMPA_I_hsp', record=True, dt=record_dt, name='g_AMPA_I_hsp')
    gGABA_mon_I = StateMonitor(P_I, 'g_GABA_I_hsp', record=True, dt=record_dt, name='g_GABA_I_hsp')

    # sp_PG = SpikeMonitor(pg_inp, name = 'sp_inp')
    sp_E = SpikeMonitor(P_E, name = 'sp_exc')
    sp_I = SpikeMonitor(P_I, name = 'sp_inh')

    net = Network(collect())
    net.run(0*ms)
    return net

class SimWrapper:
    def __init__(self):
        self.net = build_network(tau_ins_AMPA_E=1000*second,
                                 tau_int_AMPA_E=100*second,
                                 tau_ins_GABA_E=100*second,
                                 tau_int_GABA_E=1000*second)

        self.device = get_device()
        self.device.build(run=False, directory=None)
        self.N_inp = self.net['inp'].N

    def do_run(self, params):
        tau_ins_A_E, tau_int_A_E, tau_ins_G_E, tau_int_G_E = params

        from brian2.devices import device_module
        device_module.active_device = self.device

        self.net['P_E'].tau_ins_AMPA_E = tau_ins_A_E
        self.net['P_E'].tau_int_AMPA_E = tau_int_A_E
        self.net['P_E'].tau_ins_GABA_E = tau_ins_G_E
        self.net['P_E'].tau_int_GABA_E = tau_int_G_E

        kill_mask = np.ones(100, dtype=np.float32)
        #kill_mask[range(40,47)]=0 #yes
        #kill_mask[np.array([range(40,47),range(21,28)]).flatten()]=0 #yes
        kill_mask[np.array([range(43,47),range(33,37)]).flatten()]=0
        N_inp=self.net['inp'].N
        rates = np.random.normal(loc=40, scale=3, size=N_inp)
        rates *= kill_mask
        rate_img = np.clip(rates, 5, 50)*Hz
        self.net['inp'].rates = rate_img

        label = f"Ains{int(tau_ins_A_E/second)}_Aint{int(tau_int_A_E/second)}_" \
                f"Gins{int(tau_ins_G_E/second)}_Gint{int(tau_int_G_E/second)}"
        results_dir = f"results_{label}"

        sim_time = 100*second
        self.device.run(run_args={},
                        additional_run_args={'duration': sim_time},
                        results_directory=results_PATH)

        rE = self.net['rate_pop_mon_E'].r_pop_E[0, -1]
        return float(rE/Hz)



taus_ins_A = [500*second, 5000*second]
taus_int_A = [50*second, 500*second]
taus_ins_G = [500*second, 5000*second]
taus_int_G = [50*second, 500*second]

param_list = []
for tAins in taus_ins_A:
    for tAint in taus_int_A:
        for tGins in taus_ins_G:
            for tGint in taus_int_G:
                param_list.append((tAins, tAint, tGins, tGint))

sim = SimWrapper()

output_rates = [sim.do_run(p) for p in param_list]



INFO       Cache size for target 'cython': 1863 MB.
You can call clear_cache('cython') to delete all files from the cache or manually delete files in the '/home/vincent/.cython/brian_extensions' directory. [brian2]
WARNING    'tau_ins_AMPA_E' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value 1. * ksecond. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]
WARNING    'tau_ins_GABA_E' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value 100. * second. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]
WARNING    'tau_int_AMPA_E' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value 100. * second. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]
WARNING    'tau_int_GABA_E' is an internal variable of group 'neurongroup', but a